# Quantum teleportation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/tutorials/quantum-teleportation.ipynb)

Move a qubit's state across a circuit using entanglement alone, without the qubit itself travelling.

Nothing in this notebook costs anything or needs an account until the last section. The result shown is read from the public certificate for the run that actually produced it.

Full write-up: [Quantum teleportation](https://zksf.org/blog/quantum-teleportation-circuit/)


## The idea

Teleportation transfers an unknown quantum state from one qubit to another. It does not move matter, and it does not beat the speed of light: it consumes a shared entangled pair and two classical bits.

Alice holds the state to send and half of an entangled pair. She entangles her state with her half and measures both. Bob, holding the other half, applies a correction and ends up with Alice's original state.

Here Alice's qubit is prepared to read 1 about 75 percent of the time. If teleportation works, Bob's qubit, which never touched hers, should show the same split.


## The circuit


In [ ]:
!pip install -q qiskit


In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.ry(2.0944, 0)     # Alice's state: about 75% |1>

qc.h(1)              # entangled pair shared between Alice and Bob
qc.cx(1, 2)

qc.cx(0, 1)          # Alice entangles her state with her half
qc.h(0)

qc.cx(1, 2)          # Bob's correction
qc.cz(0, 2)

qc.measure(2, 0)     # measure only Bob's qubit
print(qc.draw(output='text'))


## What happened when this ran

The ry rotation is non-Clifford, so this ran on the exact statevector engine.

The cell below reads the real certificate for that run. No account, no cost.


In [ ]:
import requests

CERT = "ee40a818c5754d3b"
c = requests.get(f"https://api.zksf.org/certify/{CERT}/json", timeout=30).json()

print(f"engine   : {c['engine']}")
print(f"method   : {c['method']}")
print(f"shots    : {c['shots']}")
if c.get('expectation') is not None:
    print(f"energy   : {c['expectation']:.6f}")
for bits, n in (c.get('top_outcomes') or []):
    print(f"  {bits}  {n}")
print(f"accuracy : {c.get('error_bound', 'exact, shot noise only')}")
print(f"verify   : {c['verify_url']}")


## Reading the result

Bob measured 1 in 741 shots and 0 in 259, a 74 to 26 split. That is Alice's original 75/25 state, reproduced on a qubit at the other end of the circuit, within shot noise. The state made the trip.

That certificate is public. Anyone can open the verify link, or check it programmatically without an account:

```
pip install zcc-verify
zcc-verify ee40a818c5754d3b
```


## Run it yourself

Optional, and this part does cost. Circuits this small are a fraction of a cent, and `estimate()` prices any job for free before you commit to it. Get a token from [app.zksf.org](https://app.zksf.org).


In [ ]:
!pip install -q qsim-sdk


In [ ]:
import getpass
import qsim_sdk

client = qsim_sdk.Client(token=getpass.getpass("ZKSF API token: "))

est = client.estimate(qc, shots=1000)
print('engine:', est['engine'], '| cost: $', est['predicted_cost_usd'])


In [ ]:
job = client.run(qc, shots=1000)

print(job['result'].get('counts'))
print(job['result'].get('expectation'))
print(job['result']['error_info'])


## Next

- [The full article](https://zksf.org/blog/quantum-teleportation-circuit/), with the maths and the background
- [All tutorials](https://zksf.org/blog/) and the [glossary](https://zksf.org/glossary-of-essential-quantum-computing-terms-for-beginners/)
- [How the accuracy statements work](https://zksf.org/quantum-computing-certification/), and [the paper](https://doi.org/10.5281/zenodo.21851381)
- [Quickstart notebook](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/quickstart.ipynb): four certified runs, including one on real quantum hardware
